# Hands-On: Academic Paper Retrieval-Augmented Generation (RAG) System

### Name - Anik Chowdhury

This notebook implements an end-to-end Retrieval-Augmented Generation (RAG) system designed for academic literature search and summarization. It loads research papers from PDF files, extracts metadata (title and publication year), indexes the content in a FAISS vector database, retrieves unique relevant papers ranked by semantic similarity and recency, and generates structured summaries using a Large Language Model (LLM).

### Step 1: Import Required Libraries

We begin by importing all necessary Python packages and LangChain modules.
- **Standard & Environment Tools:** `os`, `Path`, `getpass`, and `load_dotenv` help manage file paths and secret API keys.
- **Document Loaders & Splitters:** `DirectoryLoader` and `PyPDFLoader` read PDF files from a folder, while `RecursiveCharacterTextSplitter` breaks long papers into smaller text chunks.
- **Embeddings & Vector Store:** `HuggingFaceEmbeddings` and `GoogleGenerativeAIEmbeddings` convert text into numerical vectors, and `FAISS` stores these vectors for fast similarity search.
- **LLMs & LCEL Components:** `ChatGroq` and `ChatGoogleGenerativeAI` provide access to the language models, while `ChatPromptTemplate` and `StrOutputParser` structure the prompt and clean the final output.

In [2]:
import os
from pathlib import Path
from getpass import getpass
from dotenv import load_dotenv

# LangChain Document Loaders & Text Splitters
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings & Vector Store (Multiple Options Included)
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

# Large Language Models (Groq & Google Gemini)
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI

# Core LangChain Documents, Prompts & Output Parsers
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

C:\Users\ANIK CHOWDHURY\Desktop\Tempr\ipykernel_2936\1148822580.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader


### Step 2: Load Environment Variables

We load the secret API keys (`GROQ_API_KEY` and `GOOGLE_API_KEY`) from a local `.env` file into the system environment.

**Note:** Using `load_dotenv()` is safe and will not crash the notebook if an optional API key is missing. Optional strict assertion checks are kept commented out below so you can enable them only when debugging missing keys.

In [3]:
# Load environment variables from the .env file
load_dotenv()

print("Environment variables loaded successfully.")

# Optional strict checks (Uncomment only if you want to verify specific keys in .env)
# assert os.getenv("GROQ_API_KEY") is not None, "GROQ_API_KEY Not Found"
# assert os.getenv("GOOGLE_API_KEY") is not None, "GOOGLE_API_KEY Not Found"

Environment variables loaded successfully.


### Step 3: Define and Validate the Dataset Directory

We specify the folder path (`DATA_PATH = "papers"`) where all the academic PDF files are stored. The code checks that the folder exists and counts the total number of `.pdf` files inside it.

**Note:** If the system cannot find the `papers` folder or throws a `FileNotFoundError`, please verify your folder location and provide the correct relative or absolute path in `DATA_PATH`.

In [4]:
DATA_PATH = "papers"
if not os.path.exists(DATA_PATH):
    DATA_PATH = "../../papers"


pdf_files = list(Path(DATA_PATH).rglob("*.pdf"))

if not pdf_files:
    raise FileNotFoundError("No PDF files found in the papers folder!")

print("Total PDF files:", len(pdf_files))

Total PDF files: 50


### Step 4: Load PDF Papers and Extract Metadata

In this step, we process all the PDF files in the directory and prepare one complete `Document` object per research paper:
1. **Load Pages:** `DirectoryLoader` with `PyPDFLoader` reads every page of every PDF file in the `papers` folder.
2. **Merge Pages per Paper:** Since a single research paper can span multiple pages, we group and combine all pages belonging to the same PDF file (`source`) into a single text string.
3. **Extract Title and Year Metadata:** The code searches the paper text for `"Title:"` and `"Year:"` fields. If found, it extracts the exact paper title and converts the publication year into an integer. Otherwise, it falls back to the file name and sets the year to `0`.
4. **Create Document Objects:** Each full paper is stored as a LangChain `Document` containing the combined text and a metadata dictionary (`source`, `title`, `year`).

In [5]:
loader = DirectoryLoader(
    DATA_PATH,
    glob="**/*.pdf",
    loader_cls=PyPDFLoader,
    show_progress=True,
    silent_errors=False
)

pages = loader.load()

# Combine all pages of each PDF into a single text entry
paper_text = {}

for page in pages:
    source = page.metadata["source"]
    paper_text[source] = paper_text.get(source, "") + "\n" + page.page_content

# Create one Document object for each complete paper with extracted metadata
papers = []

for source, text in paper_text.items():
    title = Path(source).stem
    year = 0

    if "Title:" in text and "Year:" in text:
        title = text.split("Title:")[1].split("Year:")[0].strip()
        year_text = text.split("Year:")[1].split("\n")[0].strip()

        try:
            year = int(year_text)
        except ValueError:
            year = 0

    papers.append(
        Document(
            page_content=text,
            metadata={
                "source": source,
                "title": title,
                "year": year
            }
        )
    )

print("Total pages loaded:", len(pages))
print("Total papers loaded:", len(papers))

100%|██████████| 50/50 [00:01<00:00, 32.24it/s]

Total pages loaded: 100
Total papers loaded: 50


### Step 5: Split Papers into Smaller Text Chunks

Academic papers are too long to pass directly into an embedding model or LLM all at once. We use `RecursiveCharacterTextSplitter` to break each paper into smaller chunks of **1000 characters** with a **100-character overlap**. 

The overlap ensures that important sentences or ideas split across two chunks do not lose their surrounding context. Each chunk automatically inherits the `source`, `title`, and `year` metadata from its parent paper.

In [6]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

chunks = splitter.split_documents(papers)

print("Total chunks:", len(chunks))

Total chunks: 399


### Step 6: Initialize the Embedding Model

An embedding model converts text chunks into numerical vectors so the system can measure semantic similarity between a user's query and the research papers. We have included two embedding model options below for flexibility:

- **Option 1: Local HuggingFace Embeddings (Default Active):** Uses `HuggingFaceEmbeddings` (`sentence-transformers/all-MiniLM-L6-v2`). It runs locally on your machine, is completely free, and does not fail due to API quota limits when indexing many PDFs.
- **Option 2: Google Gemini Embeddings (Alternative):** Uses Google's cloud embedding model (`gemini-embedding-2`) via `GOOGLE_API_KEY`.

**How to Switch Embedding Models:**
1. Add a `#` symbol at the beginning of each line of the currently active `embeddings = ...` block to comment it out.
2. Remove the `#` symbol from the beginning of the lines in the alternative block you want to use, then re-run this cell and Step 7.

In [7]:
# Option 1: Local HuggingFace Embeddings (Default Active)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Option 2: Google Gemini Embeddings (Uncomment below and comment Option 1 to use)
# embeddings = GoogleGenerativeAIEmbeddings(
#     model="gemini-embedding-2"
# )

print("Embedding model initialized successfully.")

c:\Users\ANIK CHOWDHURY\Desktop\Research Paper\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2754.85it/s]


Embedding model initialized successfully.


### Step 7: Create the FAISS Vector Store

We pass our text chunks and the initialized embedding model into **FAISS** (Facebook AI Similarity Search). FAISS generates vector embeddings for all chunks and indexes them in memory so we can perform fast similarity searches.

In [8]:
vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

print("FAISS vector store created successfully.")

FAISS vector store created successfully.


### Step 8: Define the Custom Dual-Stage Search Function

Standard vector search can return multiple chunks from the same paper and ignores how recent the research is. Our custom `search(query, k=5)` function solves both problems in three clear stages:
1. **Semantic Similarity Search:** It compares the query against all chunks in the FAISS vector store and sorts the results by similarity distance score (a lower distance score means a closer match).
2. **Deduplication by Paper:** It loops through the sorted chunks and keeps track of `seen_sources`. Only the best-matching chunk from each unique PDF file is added to `unique_papers` until we collect `k` distinct papers.
3. **Recency Sorting (Publication Year):** Finally, it sorts the `k` selected unique papers by their `year` metadata in descending order (`reverse=True`), ensuring the newest research papers appear at the top of the list.

In [9]:
def search(query, k=5):
    results = vectorstore.similarity_search_with_score(
        query,
        k=len(chunks)
    )

    # Sort by similarity score (lower distance is better)
    results.sort(key=lambda x: x[1])

    unique_papers = []
    seen_sources = set()

    for doc, score in results:
        source = doc.metadata.get("source", "")

        if source not in seen_sources:
            seen_sources.add(source)
            unique_papers.append(doc)

        if len(unique_papers) == k:
            break

    # Sort selected papers by publication year (newest first)
    unique_papers.sort(
        key=lambda doc: doc.metadata.get("year", 0),
        reverse=True
    )

    return unique_papers

### Step 9: Initialize the Large Language Model (LLM)

The Large Language Model reads the retrieved paper chunks and writes structured summaries based on our instructions. We have configured two LLM options in this cell:

- **Option 1: ChatGroq (Default Active):** Uses `openai/gpt-oss-20b` via the Groq API with `temperature=0` for factual, deterministic responses.
- **Option 2: Google Gemini (Backup Alternative):** Uses `ChatGoogleGenerativeAI` (`gemini-3.6-flash`).

**How to Switch LLMs (If Groq Fails or Hits Rate Limits):**
1. Add a `#` symbol in front of the `ChatGroq` lines to comment them out.
2. Remove the `#` symbols from the `ChatGoogleGenerativeAI` block below it and run the cell.

In [10]:
# Option 1: Groq LLM (Default Active)
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

# Option 2: Google Gemini LLM (Uncomment below and comment Option 1 if Groq fails)
# llm = ChatGoogleGenerativeAI(
#     model="gemini-3.6-flash",
#     temperature=0
# )

print("LLM initialized successfully.")

LLM initialized successfully.


### Step 10: Define the Prompt Template

We create a detailed prompt template that instructs the AI to act as an academic research assistant. To prevent hallucination and ensure consistent formatting, the prompt enforces strict rules:
- Use **only** the provided context without inventing outside facts.
- Return **all** `{paper_count}` retrieved papers without skipping or merging any paper.
- Preserve the exact title and publication year from the metadata.
- Provide a concise 2–3 sentence summary for each paper using a standardized Markdown layout.

In [11]:
template = """
You are an academic research assistant.

Answer the question using ONLY the provided context.

IMPORTANT RULES:
1. You MUST return ALL the papers provided in the context.
2. Return EXACTLY {paper_count} papers.
3. Do NOT skip, merge, or ignore any paper.
4. Keep the same paper titles and publication years as provided.
5. Write a concise 2-3 sentence summary for each paper.
6. If a paper is only partially relevant, still include it.
7. Do not invent facts that are not in the context.

Use this exact format for EVERY paper:

**Paper Number:** 1
**Title:** Paper title
**Published Year:** Publication year
**Abstract/Summary:** 2-3 sentence summary
---

Context:
{context}

Question:
{question}
"""

prompt = ChatPromptTemplate.from_template(template)

### Step 11: Build the RAG Chain using LCEL

We connect our components using **LangChain Expression Language (LCEL)** with the pipe (`|`) operator:
`rag_chain = prompt | llm | StrOutputParser()`

- First, `prompt` formats the input variables (`context`, `question`, and `paper_count`) into the final instructions.
- Next, `llm` processes those instructions and generates the response.
- Finally, `StrOutputParser()` extracts the clean text string from the model's raw message object.

In [12]:
rag_chain = prompt | llm | StrOutputParser()

### Step 12: Retrieve Relevant Papers for a Query

We test our custom `search` function with the query `"Machine Learning and Data Science"` and set `k = 6` to retrieve the top 6 unique, most relevant academic papers sorted from newest to oldest.

In [13]:
query = "Machine Learning and Data Science"
k = 6

results = search(query, k)

results

[Document(id='b309bae2-160c-49d8-9f52-fac49be16a23', metadata={'source': 'papers\\Paper-pages-19.pdf', 'title': 'On High-dimensional Feature Selection Using\nDimensionality Reduction Methods', 'year': 2024}, page_content='reliance on synthetic workloads, arguing that results on open census microdata may not generalise to urban\nmobility analysis. In contrast to these efforts, our work treats high-dimensional feature selection and causal\neffect estimation from observational data jointly rather than as independent concerns. Despite this body of\nMock Academic Dataset - synthetic content for RAG evaluation Page 38\nliterature, no prior study has systematically compared dimensionality reduction methods and gradient-boosted\ndecision trees under identical experimental conditions.\n3. Methodology\nWe formalise high-dimensional feature selection as a constrained optimisation task in which the objective is to\nmaximise forecast MAPE while keeping AUC-ROC within a prescribed budget. The propos

### Step 13: Format Context and Generate the Final Answer

In this final step, we combine the `Title`, `Year`, and `Content` of each retrieved document into a single formatted `context` string. We then invoke `rag_chain` with the user's question, the formatted context, and the exact number of retrieved papers (`len(results)`), and print the structured summary report.

In [14]:
context = "\n\n".join(
    f"Title: {doc.metadata['title']}\n"
    f"Year: {doc.metadata['year']}\n"
    f"Content: {doc.page_content}"
    for doc in results
)

answer = rag_chain.invoke({
    "question": query,
    "context": context,
    "paper_count": len(results)
})

print(answer)

**Paper Number:** 1  
**Title:** On High-dimensional Feature Selection Using Dimensionality Reduction Methods  
**Published Year:** 2024  
**Abstract/Summary:** This study jointly tackles high‑dimensional feature selection and causal effect estimation from observational data, formulating feature selection as a constrained optimization that maximizes forecast MAPE while keeping AUC‑ROC within a specified budget. It systematically compares dimensionality reduction techniques and gradient‑boosted decision trees under identical experimental conditions, noting that reliance on synthetic workloads may limit generalizability to urban mobility analysis.  

---  

**Paper Number:** 2  
**Title:** Rethinking Concept Drift Adaptation: A Dimensionality Reduction Methods Perspective  
**Published Year:** 2023  
**Abstract/Summary:** The authors propose a framework that uses dimensionality reduction to adapt to concept drift, explicitly addressing reproducibility of analytical workflows. Evaluated o